# 🧠 Phase 4: Autonomous Closed-Loop Fuzzy DEMATEL & Causal Triangulation
## *Task-Technology Fit Analysis of Modern AI-Driven Intrusion Detection: An Axiomatic-Empirical Fuzzy DEMATEL Simulation Framework*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

---

### 📌 Scientific Objectives:
1. **Triangular Fuzzy DEMATEL Formulation**:
   - Model the 8 architectural-empirical causal factors ($F_1$ Feature Topology, $F_2$ In-Context Memory, $F_3$ Inference Latency, $F_4$ Memory Footprint, $F_5$ Zero-Day Generalization, $F_6$ Throughput Scalability, $F_7$ Data Decontamination, $F_8$ TTF Alignment).
   - Compute normalized fuzzy direct relation matrix $\tilde{X}$ and invert $(I - \tilde{X})$ to obtain total relation matrix $\tilde{T}$.
   - Apply CFCS (Converting Fuzzy data into Crisp Scores) defuzzification to determine Prominence ($D + R$) and Net Cause-Effect Relation ($D - R$).
2. **10,000-Iteration Monte Carlo Stability Proof**:
   - Perturb fuzzy boundary limits under Gaussian noise ($\sigma = 0.05$) to mathematically prove topological rank invariance (Kendall's $W \ge 0.95$).
3. **DirectLiNGAM Causal Triangulation**:
   - Ingest empirical telemetry from Phase 2 / Phase 3 and discover non-Gaussian causal Directed Acyclic Graphs (DAGs).
   - Prove algorithmic triangulation by bounding Structural Hamming Distance ($SHD \le 2$).
4. **Publication Causal Digraph & Quadrant Map**:
   - Render high-resolution Prominence-Relation scatter maps and NetworkX causal digraphs.


### 1. ☁️ Google Drive Mount & Project Root Auto-Resolution


In [ ]:
import os, sys
from pathlib import Path

# 0. Enable automatic reloading of modified modules in Colab / Jupyter
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

# 1. Mount Google Drive if running in Colab
try:
    from google.colab import drive
    if not Path('/content/drive').exists() and not Path('/content/My Drive').exists():
        drive.mount('/content/drive')
except ImportError:
    print("ℹ️ Running in local/workstation environment.")

# 2. Candidate root paths (supporting both 'Colab Notebook' and 'Colab Notebooks')
CANDIDATE_ROOTS = [
    Path('/content/drive/MyDrive/Colab Notebooks'),
    Path('/content/drive/My Drive/Colab Notebooks'),
    Path('/content/drive/MyDrive/Colab Notebook'),
    Path('/content/drive/My Drive/Colab Notebook'),
    Path('/Colab Notebooks'),
    Path('/Colab Notebook'),
    Path('/content/My Drive/Colab Notebooks'),
    Path('/content/My Drive/Colab Notebook'),
    Path('/content/Colab Notebooks'),
    Path('/content/Colab Notebook'),
    Path('/content/drive/MyDrive/is_ai-vuln'),
    Path('/content/drive/My Drive/is_ai-vuln'),
    Path('/content/is_ai-vuln'),
    Path('.').resolve()
]

PROJECT_ROOT = None
for cand in CANDIDATE_ROOTS:
    if cand.exists() and ((cand / 'src').exists() or (cand / 'data' / 'raw').exists()):
        PROJECT_ROOT = cand.resolve()
        break

# Dynamic discovery inside /content/drive if not yet matched
if PROJECT_ROOT is None and Path('/content/drive').exists():
    for drive_parent in [Path('/content/drive/MyDrive'), Path('/content/drive/My Drive'), Path('/content/drive'), Path('/content/My Drive')]:
        if drive_parent.exists():
            try:
                for sub in drive_parent.iterdir():
                    if sub.is_dir() and ('colab notebook' in sub.name.lower() or 'is_ai-vuln' in sub.name.lower()):
                        if (sub / 'src').exists() or (sub / 'data' / 'raw').exists():
                            PROJECT_ROOT = sub.resolve()
                            break
            except Exception:
                pass
            if PROJECT_ROOT:
                break

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path('.').resolve()

os.chdir(str(PROJECT_ROOT))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 3. Locate authentic dataset raw storage directory across candidate paths
DATA_RAW_DIR = None
KNOWN_SUBFOLDERS = ['cic-ddos2019', 'machinelearningcve', 'nsl-kdd', 'ton-iot', 'trafficlabelling', 'unsw-data-full']

for cand_raw in [
    Path('/content/drive/MyDrive/Colab Notebooks/data/raw'),
    Path('/content/drive/My Drive/Colab Notebooks/data/raw'),
    Path('/content/drive/MyDrive/Colab Notebook/data/raw'),
    Path('/content/drive/My Drive/Colab Notebook/data/raw'),
    Path('/Colab Notebooks/data/raw'),
    Path('/Colab Notebook/data/raw'),
    PROJECT_ROOT / 'src' / 'data' / 'actual-data',
    PROJECT_ROOT / 'actual-data',
    PROJECT_ROOT / 'data' / 'raw',
]:
    if cand_raw.exists():
        try:
            sub_names = [c.name.lower() for c in cand_raw.iterdir() if c.is_dir()]
            if any(k in sub_names for k in KNOWN_SUBFOLDERS):
                DATA_RAW_DIR = cand_raw.resolve()
                break
        except Exception:
            pass

# Dynamic search inside /content/drive if not yet matched
if DATA_RAW_DIR is None and Path('/content/drive').exists():
    for drive_parent in [Path('/content/drive/MyDrive'), Path('/content/drive/My Drive'), Path('/content/drive'), Path('/content/My Drive')]:
        if drive_parent.exists():
            try:
                for sub in drive_parent.iterdir():
                    if sub.is_dir() and ('colab notebook' in sub.name.lower() or 'is_ai-vuln' in sub.name.lower()):
                        cand = sub / 'data' / 'raw'
                        if cand.exists():
                            sub_names = [c.name.lower() for c in cand.iterdir() if c.is_dir()]
                            if any(k in sub_names for k in KNOWN_SUBFOLDERS):
                                DATA_RAW_DIR = cand.resolve()
                                break
            except Exception:
                pass
            if DATA_RAW_DIR:
                break

if DATA_RAW_DIR is None:
    DATA_RAW_DIR = (PROJECT_ROOT / 'data' / 'raw').resolve()

# Link local data/raw to Drive data/raw if different (Colab Linux filesystem)
local_raw = PROJECT_ROOT / 'data' / 'raw'
if DATA_RAW_DIR.exists() and local_raw.resolve() != DATA_RAW_DIR.resolve():
    if not local_raw.exists():
        try:
            local_raw.parent.mkdir(parents=True, exist_ok=True)
            local_raw.symlink_to(DATA_RAW_DIR, target_is_directory=True)
            print(f"🔗 Linked: {local_raw} -> {DATA_RAW_DIR}")
        except Exception:
            pass

detected_folders = [f.name for f in DATA_RAW_DIR.iterdir() if f.is_dir()] if DATA_RAW_DIR.exists() else []

print("=" * 80)
print(f"✅ Active Project Root : {PROJECT_ROOT}")
print(f"✅ src/ directory found: {(PROJECT_ROOT / 'src').exists()}")
print(f"📁 Active Raw Data Path: {DATA_RAW_DIR}")
print(f"🔍 Detected Raw Folders: {detected_folders}")
known_real = ['CIC-DDoS2019', 'MachineLearningCVE', 'NSL-KDD', 'TON-IoT', 'ToN-IOT', 'TrafficLabelling', 'unsw-data-full']
found_known = [f for f in detected_folders if f in known_real]
if found_known:
    print(f"🛡️ [DATA STATUS: REAL BENCHMARK DATASETS DETECTED] Found: {found_known}")
else:
    print("ℹ️ [DATA STATUS] Datasets will be dynamically located across Drive paths.")
print("=" * 80)


### 2. 🧮 Autonomous Triangular Fuzzy DEMATEL Execution

Executes fuzzy matrix inversion $\tilde{T} = \tilde{X}(I - \tilde{X})^{-1}$, CFCS defuzzification, and dynamic thresholding $\alpha = \mu + \beta \cdot \sigma$.


In [ ]:
import pandas as pd
from src.dematel import (
    run_closed_loop_fuzzy_dematel,
    run_monte_carlo_sensitivity_proof,
    run_causal_triangulation,
    plot_causal_network_digraph
)

dematel_output = run_closed_loop_fuzzy_dematel(beta=0.50)
df_causal = pd.DataFrame(dematel_output["summary_table"])
print(f"✅ Fuzzy DEMATEL Solved! Alpha Threshold: {dematel_output['alpha_threshold']:.4f}")
print("📊 Causal Prominence (D+R) and Net Relation (D-R) Index Table:")
display(df_causal)


### 3. 🎲 10,000-Iteration Monte Carlo Robustness Proof ($W \ge 0.95$)

Perturbs fuzzy boundary limits across 10,000 Monte Carlo iterations to verify ranking stability under subjective uncertainty.


In [ ]:
L = dematel_output["fuzzy_bounds"]["L"]
M = dematel_output["fuzzy_bounds"]["M"]
U = dematel_output["fuzzy_bounds"]["U"]

mc_res = run_monte_carlo_sensitivity_proof(L, M, U, n_iterations=10000, noise_sigma=0.05)
print(f"🎲 Monte Carlo Sensitivity Proof (10,000 Iterations):")
print(f" - Kendall's W (Prominence): {mc_res['kendalls_w_prominence']} (p = {mc_res['prominence_p_value']:.4e})")
print(f" - Kendall's W (Relation)  : {mc_res['kendalls_w_relation']}")
print(f" - Q1 Stability Target Met : {mc_res['stability_target_met']} (Kendall's W >= 0.95)")


### 4. 🔗 Algorithmic Causal Triangulation (DirectLiNGAM)

Integrates empirical telemetry vectors from Phase 2 benchmark runs and evaluates Structural Hamming Distance ($SHD$).


In [ ]:
from src.dematel.empirical_mapper import extract_empirical_telemetry_from_experiments

telemetry, is_synthetic_telemetry = extract_empirical_telemetry_from_experiments(
    experiment_dir=PROJECT_ROOT / "experiment_output",
    n_folds=5
)

triangulation_res = run_causal_triangulation(telemetry, dematel_output["adjacency_matrix"])

print(f"🔗 Causal Triangulation Status:")
print(f" - Telemetry Provenance: {'REFERENCE TELEMETRY' if is_synthetic_telemetry else 'REAL EMPIRICAL BENCHMARKS'}")
print(f" - Discovery Algorithm : {triangulation_res['triangulation_method']}")
print(f" - Structural Hamming Distance (SHD): {triangulation_res['structural_hamming_distance']} (Target Ceiling <= {triangulation_res['target_shd_ceiling']})")
print(f" - Triangulation Validated: {triangulation_res['triangulation_passed']} (Axiomatic and Empirical models converged)")


### 5. 🕸️ Publication Causal Network Digraph & Quadrant Scatter Map

Renders the Prominence-Relation scatter map (Cause vs. Effect quadrants) and directed causal influence graph.


In [ ]:
import matplotlib.pyplot as plt

out_dir = PROJECT_ROOT / "experiment_output" / "fuzzy_dematel"
out_dir.mkdir(parents=True, exist_ok=True)

fig_digraph = plot_causal_network_digraph(
    dematel_output, output_filepath=str(out_dir / "figure_causal_network_digraph")
)
plt.show()
print(f"💾 Causal digraph rendered and saved to: {out_dir / 'figure_causal_network_digraph.png'}")
